In [5]:
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# 1. 데이터 로드
df = pd.read_csv('drugs.csv')
smiles_list = df['SMILES'].dropna().tolist()
total_drugs = len(smiles_list)

# 2. 최적화된 핵심 SMARTS 패턴 정의
# (a) 가점 구조 (필수 뼈대 및 물성 개선)
pos_patterns = {
    'Benzene (방향족 뼈대)': 'c1ccccc1',
    'Amide (단백질 결합)': 'C(=O)N',
    'Piperazine (용해도 개선)': 'N1CCNCC1'
}

# (b) 감점 구조 (치명적 독성 및 물성 저하)
neg_patterns = {
    'Peroxide (반응성 독성)': 'OO',
    'Epoxide (유전자 독성)': 'C1OC1',
    'Quinone (세포 및 DNA 독성)': 'O=C1C=CC(=O)C=C1',
    'Hydrazine (심각한 간 독성)': 'NN'
}

def analyze_patterns(patterns_dict):
    results = {}
    for name, smarts in patterns_dict.items():
        pattern = Chem.MolFromSmarts(smarts)
        count = sum(1 for s in smiles_list if Chem.MolFromSmiles(s) and Chem.MolFromSmiles(s).HasSubstructMatch(pattern))
        results[name] = count
    return results

# 3. 결과 출력
print("-" * 50)
print("최적화된 핵심 SMARTS 패턴 출현 빈도 검증")
print("-" * 50)

print("[가점 대상 구조 (Positive)]")
for name, count in analyze_patterns(pos_patterns).items():
    print(f"{name:<25}: {(count / total_drugs) * 100:>5.1f}% ({count}개)")

print("\n[감점/배제 대상 구조 (Negative)]")
for name, count in analyze_patterns(neg_patterns).items():
    print(f"{name:<25}: {(count / total_drugs) * 100:>5.1f}% ({count}개)")
print("-" * 50)

--------------------------------------------------
최적화된 핵심 SMARTS 패턴 출현 빈도 검증
--------------------------------------------------
[가점 대상 구조 (Positive)]
Benzene (방향족 뼈대)         :  66.1% (9999개)
Amide (단백질 결합)           :  35.1% (5303개)
Piperazine (용해도 개선)      :   4.3% (643개)

[감점/배제 대상 구조 (Negative)]
Peroxide (반응성 독성)        :   0.2% (35개)
Epoxide (유전자 독성)         :   0.6% (96개)
Quinone (세포 및 DNA 독성)    :   0.2% (24개)
Hydrazine (심각한 간 독성)     :   1.6% (249개)
--------------------------------------------------
